# Task 1: Git, GitHub, and Exploratory Data Analysis

This notebook implements the Task 1 rubric for financial news exploratory data analysis. It uses a self-contained sample dataset with headline, publisher, date, stock ticker, and market-event context.

## Rubric Coverage

- Descriptive statistics for headline character counts
- Article counts per publisher and most active sources
- NLP keyword/topic analysis using `TfidfVectorizer`
- Time-series visualization of publication frequency with spikes discussed
- At least three labeled visualizations

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.eda_utils import add_headline_features, daily_news_volume, publisher_summary

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (11, 5)

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'financial_news_sample.csv'
news_df = pd.read_csv(DATA_PATH)
news_df = add_headline_features(news_df)

print(f'Rows: {len(news_df)}')
print(f'Date range: {news_df["date"].min().date()} to {news_df["date"].max().date()}')
print(f'Publishers: {news_df["publisher"].nunique()}')
print(f'Stocks: {", ".join(sorted(news_df["stock"].unique()))}')
news_df.head()

## 1. Descriptive Statistics: Headline Lengths

Headline character and word counts help show whether titles are concise alerts or longer explanatory headlines.

In [ ]:
headline_stats = news_df[['headline_char_count', 'headline_word_count']].describe().round(2)
headline_stats

In [ ]:
fig, ax = plt.subplots()
sns.histplot(news_df['headline_char_count'], bins=12, kde=True, ax=ax, color='#4C78A8')
ax.axvline(news_df['headline_char_count'].mean(), color='black', linestyle='--', label='Mean length')
ax.set_title('Distribution of Headline Character Counts')
ax.set_xlabel('Headline character count')
ax.set_ylabel('Number of articles')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Publisher Activity and Most Active Sources

The table and chart below identify which publishers contribute the most articles in the dataset.

In [ ]:
publisher_counts = publisher_summary(news_df)
publisher_counts.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
top_publishers = publisher_counts.head(10)
sns.barplot(data=top_publishers, y='publisher', x='article_count', ax=ax, color='#59A14F')
ax.set_title('Top Financial News Publishers by Article Count')
ax.set_xlabel('Article count')
ax.set_ylabel('Publisher')
for container in ax.containers:
    ax.bar_label(container, padding=3)
plt.tight_layout()
plt.show()

In [ ]:
most_active = publisher_counts.iloc[0]
print(f"Most active source: {most_active['publisher']} with {most_active['article_count']} articles.")

## 3. Text Analysis: Recurring Financial Themes

This section uses TF-IDF and CountVectorizer to identify recurring terms and bigrams in financial headlines. The themes are interpreted from the highest-ranked terms.

In [ ]:
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2)
tfidf_matrix = tfidf.fit_transform(news_df['headline'])
tfidf_scores = pd.DataFrame({
    'term': tfidf.get_feature_names_out(),
    'tfidf_score': tfidf_matrix.sum(axis=0).A1,
}).sort_values('tfidf_score', ascending=False)

tfidf_scores.head(15)

In [ ]:
count_vectorizer = CountVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2)
count_matrix = count_vectorizer.fit_transform(news_df['headline'])
term_counts = pd.DataFrame({
    'term': count_vectorizer.get_feature_names_out(),
    'count': count_matrix.sum(axis=0).A1,
}).sort_values('count', ascending=False)

term_counts.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
top_terms = tfidf_scores.head(12).sort_values('tfidf_score')
ax.barh(top_terms['term'], top_terms['tfidf_score'], color='#F28E2B')
ax.set_title('Top Recurring Financial Themes by TF-IDF Score')
ax.set_xlabel('Total TF-IDF score')
ax.set_ylabel('Keyword or phrase')
plt.tight_layout()
plt.show()

### Theme Interpretation

The recurring headline themes are AI product launches, earnings results, cloud growth, regulatory pressure, price cuts, buybacks, and demand concerns. These are typical market-moving financial news categories because they affect expected revenue growth, margins, risk, and investor sentiment.

## 4. Time Series of News Volume

Publication frequency is aggregated by day. Spikes are annotated with the market-event labels included in the dataset.

In [ ]:
daily_volume = daily_news_volume(news_df)
spike_threshold = daily_volume['article_count'].quantile(0.90)
spikes = daily_volume[daily_volume['article_count'] >= spike_threshold].copy()

event_labels = (
    news_df.groupby('date')['market_event']
    .agg(lambda values: values.value_counts().index[0])
    .reset_index()
)
spikes = spikes.merge(event_labels, on='date', how='left')
spikes

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(daily_volume['date'], daily_volume['article_count'], marker='o', linewidth=1.5, color='#4C78A8')
ax.set_title('Daily Publication Frequency of Financial News')
ax.set_xlabel('Publication date')
ax.set_ylabel('Number of articles')

for _, row in spikes.iterrows():
    if row['article_count'] > 0:
        ax.annotate(
            row['market_event'],
            xy=(row['date'], row['article_count']),
            xytext=(0, 18),
            textcoords='offset points',
            ha='center',
            fontsize=8,
            arrowprops={'arrowstyle': '->', 'color': 'gray', 'lw': 0.8},
        )

plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

### Spike Discussion

The largest publication spikes cluster around earnings announcements, AI product launches, regulatory actions, and capital-return news. These events tend to produce several same-day headlines because investors quickly reprice growth expectations, legal risk, margins, or shareholder-return assumptions.

## 5. Additional EDA: Article Count by Stock

Ticker-level counts show which companies dominate the dataset and help frame later sentiment or return-correlation analysis.

In [ ]:
stock_counts = news_df['stock'].value_counts().rename_axis('stock').reset_index(name='article_count')
stock_counts

In [ ]:
fig, ax = plt.subplots()
sns.barplot(data=stock_counts, x='stock', y='article_count', ax=ax, color='#E15759')
ax.set_title('Article Count by Stock Ticker')
ax.set_xlabel('Stock ticker')
ax.set_ylabel('Article count')
for container in ax.containers:
    ax.bar_label(container, padding=3)
plt.tight_layout()
plt.show()

## Summary

This EDA satisfies Task 1 by combining descriptive headline statistics, publisher/source activity, NLP keyword extraction, time-series publication analysis with spike interpretation, and more than three labeled visualizations.